# SDS — Report & Site

Builds the benchmark site assets and inspector from NER results.

**Requires:** `entity_matches.csv` in `DATA_DIR` (produced by `ner.ipynb`)

## 1. Configuration

In [ ]:
import sys
from pathlib import Path

PROJECT = Path(".").resolve()
if not (PROJECT / "synthdoc").exists():
    PROJECT = PROJECT.parent
DATA_DIR = PROJECT / "data"
SITE_DIR = PROJECT.parent / "sdd-benchmark-site"

sys.path.insert(0, str(PROJECT))

print(f"PROJECT:  {PROJECT}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"SITE_DIR: {SITE_DIR}")
print(f"entity_matches.csv exists: {(DATA_DIR / 'entity_matches.csv').exists()}")

## 1b. Rebuild entity_matches.csv from saved NER JSON files

In [ ]:
# Rebuilds entity_matches.csv from existing label + NER JSON files — no GPU needed
import json
import pandas as pd
from synthdoc.bench.helpers import load_ner_pages, FULL_ENTITY_TYPES
from synthdoc.bench.static import per_entity_matches
from synthdoc.bench.helpers import detailed_match

NER_DIR = DATA_DIR / "ner"
OCR_ENGINE = "glm"
NER_ENGINE = "qwen3"
ner_key = f"ner_{OCR_ENGINE}"
ocr_key = f"{OCR_ENGINE}_ocr"

# Load pages with GT entities
pages = load_ner_pages(DATA_DIR, entity_types=FULL_ENTITY_TYPES)
print(f"Loaded {len(pages)} pages")

# Load NER results from JSON files
loaded = 0
for p in pages:
    path = NER_DIR / f"{p['stem']}_{NER_ENGINE}.json"
    if path.exists():
        p[ner_key] = json.loads(path.read_text())["entities"]
        loaded += 1
    else:
        p[ner_key] = []

print(f"Loaded NER results for {loaded}/{len(pages)} pages")

# Build entity_matches.csv
model_label = f"{OCR_ENGINE}_{NER_ENGINE}"
entity_df = per_entity_matches(pages, [(model_label, ner_key)])
entity_df.to_csv(DATA_DIR / "entity_matches.csv", index=False)
print(f"Saved entity_matches.csv ({len(entity_df)} rows)")

## 2. Build Gallery Assets

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, str(SITE_DIR / "build_assets.py"), "--run", str(DATA_DIR)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

## 3. Build Inspector

In [ ]:
result = subprocess.run(
    [sys.executable, str(SITE_DIR / "build_inspector.py"), "--data-dir", str(DATA_DIR)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

## 4. Summary

In [ ]:
import pandas as pd

csv_path = DATA_DIR / "entity_matches.csv"
df = pd.read_csv(csv_path)

print(f"Loaded {len(df)} rows from entity_matches.csv")
print(f"Columns: {list(df.columns)}\n")

if "strict_f1" in df.columns and "ocr" in df.columns:
    std = df[df["entity_set"] == "standard"] if "entity_set" in df.columns else df
    summary = std.groupby("ocr").agg({"strict_f1": "mean", "relaxed_f1": "mean"}).round(4)
    print(summary.to_string())
else:
    print(df.head())

## 5. Open Site

In [ ]:
index = SITE_DIR / "index.html"
print(f"Site ready at: {index}")

# Uncomment to open in browser:
# import webbrowser
# webbrowser.open(index.as_uri())